# MOEA/D — Run missing tuning only (NSGA-II aligned)

**Goal:** Add the missing 12 × 3 scikit-learn search, 24-trial Optuna, top-3 reseeds and baseline. Reuse completed Exp 1–6. You can run this in a fresh Colab session or in the same live session as the earlier notebook.


In [ ]:
%pip -q install numpy pandas scikit-learn optuna matplotlib


In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, os, sys, subprocess, json

# Prefer an already extracted project in the current Colab session.
projects=[p.parent for p in Path('/content').rglob('moead_core.py') if 'MOEAD_Complete_Project' in p.parts and '__pycache__' not in p.parts and (p.parent/'verify_tuning_outputs.py').is_file() and (p.parent/'plot_optuna_search.py').is_file()]
# Reuse an extracted project only if it includes the updated auditing scripts.
projects=[p for p in projects if (p/'verify_tuning_outputs.py').is_file() and (p/'plot_optuna_search.py').is_file()]
if not projects:
    print('Upload MOEAD_Full_Aligned_With_Results.zip (contains completed Exp 1–6 and the tuning verification scripts).')
    uploaded=files.upload()
    project_archives=[]
    for name in uploaded:
        if not name.lower().endswith('.zip'):continue
        with zipfile.ZipFile('/content/'+name) as z:
            if any(n.endswith('/moead_core.py') for n in z.namelist()):project_archives.append(name)
    assert len(project_archives)==1, 'Upload a single MOEA/D project ZIP containing moead_core.py'
    with zipfile.ZipFile('/content/'+project_archives[0]) as z:z.extractall('/content/moead_tuning_workspace')
    projects=[p.parent for p in Path('/content/moead_tuning_workspace').rglob('moead_core.py')]
assert projects,'Project not found'
assert (projects[0]/'verify_tuning_outputs.py').is_file(), 'Please upload the updated MOEAD_Full_Aligned_With_Results.zip rather than the old project archive'
project=next((p for p in projects if (p/'results/exp6_moead/summary.json').is_file()), projects[0])
os.chdir(project)
if not Path('synthetic_fraud_500k.csv').is_file():
    print('Upload the matching synthetic_fraud_500k.csv separately (this share ZIP excludes the raw dataset)')
    files.upload()
assert Path('synthetic_fraud_500k.csv').is_file(), 'Project CSV missing'
assert Path('moead_search.py').is_file() and Path('moead_optuna.py').is_file()
print('Working project:',project)
print('Exp folders already present:',len(list(Path('results').glob('exp*_moead/summary.json'))))


## A. Run scikit-learn-style search (12 configurations × 3 seeds)
This is the same algorithm-agnostic `param_search.py` implementation as NSGA-II. Existing partial results resume automatically.


In [ ]:
subprocess.run([sys.executable,'moead_search.py','--csv','synthetic_fraud_500k.csv',
    '--n_iter','12','--seeds','3','--pop','200','--jobs','2',
    '--out','results/parameter_search/moead_search'],check=True)


## B. Run Optuna (24 trials + top three on seeds 101–103)
The SQLite study resumes if partially complete. The output is only audited as complete when all 24 trials succeed.


In [ ]:
subprocess.run([sys.executable,'moead_optuna.py','--csv','synthetic_fraud_500k.csv',
    '--trials','24','--pop','200','--workers','2','--out','results/optuna'],check=True)


## C. Exp 4 baseline on the same seeds, and Optuna search plot


In [ ]:
subprocess.run([sys.executable,'moead_optuna_baseline.py','--csv','synthetic_fraud_500k.csv',
    '--pop','200','--gens','250','--cx','0.9','--mut','0.8',
    '--out','results/optuna/exp4_setting_same_seeds.csv'],check=True)
subprocess.run([sys.executable,'plot_optuna_search.py'],check=True)


## Shared Random Forest benchmark (same NSGA-II code and 500K dataset)
The RF is a **shared model-independent benchmark** for NSGA-II, SPEA2 and MOEA/D, not a separate MOEA/D algorithm. This exact script produces `rf_benchmark_results.csv` and `rf_feature_importance.csv`. Existing results are reused.


In [ ]:
rf_dir=Path('results/rf_benchmark')
rf_dir.mkdir(parents=True,exist_ok=True)
if not (rf_dir/'rf_benchmark_results.csv').is_file() or not (rf_dir/'rf_feature_importance.csv').is_file():
    subprocess.run([sys.executable,str(Path.cwd()/'rf_benchmark.py'),str(Path.cwd()/'synthetic_fraud_500k.csv')],cwd=rf_dir,check=True)
else:
    print('RF benchmark already present — not retraining')


## D. Validate before exporting
This must print **TUNING AUDIT: PASS**. No fabricated or placeholder outputs are used.


In [ ]:
subprocess.run([sys.executable,'verify_completion.py','--csv','synthetic_fraud_500k.csv'],check=True)
subprocess.run([sys.executable,'verify_tuning_outputs.py','results'],check=True)


## E. Download complete results and source archives
Raw 500K CSV is omitted only from the code-plus-results review archive. Results ZIP contains all six experiments and tuning data.


In [ ]:
from shutil import make_archive
from zipfile import ZipFile, ZIP_DEFLATED
subprocess.run([sys.executable,'verify_tuning_outputs.py','results'],check=True)
result_zip=make_archive('/content/MOEAD_results_COMPLETE','zip',root_dir=Path.cwd(),base_dir='results')
files.download(result_zip)
full_zip=Path('/content/MOEAD_code_and_results_COMPLETE.zip')
with ZipFile(full_zip,'w',compression=ZIP_DEFLATED) as z:
    for p in Path.cwd().rglob('*'):
        if p.is_file() and p.name!='synthetic_fraud_500k.csv' and '__pycache__' not in p.parts and '.ipynb_checkpoints' not in p.parts:
            z.write(p,arcname=str(Path('MOEAD_Complete_Project')/p.relative_to(Path.cwd())))
files.download(str(full_zip))
